In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import joblib

PROJ = Path.cwd().parents[0]   # notebooks -> project root
DATA_RAW = PROJ / "data" / "raw"
DATA_PROCESSED = PROJ / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

CSV_PATH = DATA_RAW / "Preprocessed_Depression_Dataset.csv"
print("Using:", CSV_PATH)


Using: d:\Git\DepressionLevel\DepressionModelProject\data\raw\Preprocessed_Depression_Dataset.csv


In [2]:
df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)
df.head()

target_col = "Depression Level"
print("Target:", target_col)

y = df[target_col]
X = df.drop(columns=[target_col])


Shape: (501, 14)
Target: Depression Level


In [3]:
# quick inspection of data types
cat_cols = [c for c in X.columns if X[c].dtype == "object"]
num_cols = [c for c in X.columns if X[c].dtype != "object"]
print("Categorical:", cat_cols)
print("Numeric:", num_cols)


Categorical: ['Timestamp', 'What is Your Age group?', 'What is Your Gender', '1. Have You Been Feeling Sad Most of The Time During The Day, Especially Over The Last Two Weeks ?', '2. Do You Feel Unhappy Even When You Do Your Favorite Activities ?', '3. Do You Feel Tired All The Time , Do You Find It Difficulty to Complete Day Today Usual Activity ?', '4. Do you often catch yourself getting distracted while working on something important?', "5. Do you often feel that you're not as capable as your friends, even when you try your best?", "6. Do you often feel guilty about things that aren't your fault or feel worthless?", '7. Do you feel as if things won’t get better no matter what you do?', '8. Lately, have you felt so overwhelmed or hopeless that you thought about giving up on everything and even Life?', '9. Have you been having trouble falling asleep, waking early, or sleeping too much?', '10. Have you lost your appetite or found that you’re eating much less than usual?']
Numeric: []


In [4]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")


Train: (350, 13), Val: (75, 13), Test: (76, 13)


In [5]:
# numeric pipeline
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# categorical pipeline
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# combine them
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols)
])

# fit on train only
preprocessor.fit(X_train)


,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


In [6]:
X_train_p = preprocessor.transform(X_train)
X_val_p = preprocessor.transform(X_val)
X_test_p = preprocessor.transform(X_test)

# convert sparse to dense for saving (small datasets only)
if hasattr(X_train_p, "toarray"):
    X_train_p = X_train_p.toarray()
    X_val_p = X_val_p.toarray()
    X_test_p = X_test_p.toarray()

np.save(DATA_PROCESSED / "X_train.npy", X_train_p)
np.save(DATA_PROCESSED / "X_val.npy", X_val_p)
np.save(DATA_PROCESSED / "X_test.npy", X_test_p)
y_train.to_csv(DATA_PROCESSED / "y_train.csv", index=False)
y_val.to_csv(DATA_PROCESSED / "y_val.csv", index=False)
y_test.to_csv(DATA_PROCESSED / "y_test.csv", index=False)

# save the fitted transformer to reuse later
joblib.dump(preprocessor, PROJ / "models" / "preprocessor.joblib")
print("Preprocessing complete and saved.")


Preprocessing complete and saved.
